## Part 0 — Setup (3 min)

Python is the dominant language for deep learning because of libraries such as **TensorFlow/Keras** and **PyTorch** (see the *Programming Language* slides). In this lab we use **Keras**, the high-level API in TensorFlow.

*Optional:* in Colab, go to **Runtime → Change runtime type → T4 GPU** for faster training. Everything below also runs on CPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility: same random numbers every time we run
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU") or "No (CPU is fine)")

## Part 1 — A single neuron from scratch (12 min)

Before using any library, let us build the smallest possible "network": **one neuron**. It contains every mathematical element from the lecture:

| Element | Symbol | Role |
|---|---|---|
| Weights | $\mathbf{w}$ | strength of each input connection |
| Bias | $b$ | shifts the neuron's output, adds flexibility |
| Activation | $\sigma(z)$ | decides how much signal passes forward |
| Loss | $L$ | measures how wrong the prediction is |
| Optimizer | gradient descent | updates $\mathbf{w}, b$ to reduce $L$ |

$$z = \mathbf{w}^\top \mathbf{x} + b, \qquad \hat{y} = \sigma(z) = \frac{1}{1+e^{-z}}$$

### Scenario
We want to predict whether a student **passes (1)** or **fails (0)** an exam from two inputs: **study hours** and **sleep hours** (synthetic data).

In [ ]:
rng = np.random.default_rng(0)
n = 200
X_raw = rng.uniform(0, 10, size=(n, 2))           # columns: [study hours, sleep hours]
y = ((0.6 * X_raw[:, 0] + 0.4 * X_raw[:, 1] + rng.normal(0, 0.8, n)) > 5).astype(float)

X = X_raw / 10.0                                  # scale inputs to [0, 1] for stable training

plt.figure(figsize=(5, 4))
plt.scatter(X_raw[y == 0, 0], X_raw[y == 0, 1], c="tab:red", label="Fail (0)", alpha=0.7)
plt.scatter(X_raw[y == 1, 0], X_raw[y == 1, 1], c="tab:green", label="Pass (1)", alpha=0.7)
plt.xlabel("Study hours"); plt.ylabel("Sleep hours"); plt.legend(); plt.title("Training data")
plt.show()
print("X shape:", X.shape, "| y shape:", y.shape, "| pass rate:", y.mean().round(2))

### 1.1 Activation functions
The activation function adds **non-linearity**. Without it, stacking layers would still give a linear model. Here are the three most common ones.

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)
def tanh(z):    return np.tanh(z)

z = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, f, name in zip(axes, [sigmoid, relu, tanh], ["Sigmoid", "ReLU", "Tanh"]):
    ax.plot(z, f(z), lw=2); ax.axhline(0, c="gray", lw=0.5); ax.axvline(0, c="gray", lw=0.5)
    ax.set_title(name); ax.set_xlabel("z")
plt.tight_layout(); plt.show()

### 1.2 Forward pass and loss with *random* weights
For binary classification we use the **binary cross-entropy** loss:

$$L = -\frac{1}{n}\sum_{i=1}^{n}\Big[y_i\log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\Big]$$

In [ ]:
def forward(X, w, b):
    z = X @ w + b          # weighted sum + bias
    return sigmoid(z)      # activation

def bce_loss(y, y_hat, eps=1e-9):
    return -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))

w = rng.normal(0, 1, size=2)   # random starting weights
b = 0.0
y_hat = forward(X, w, b)
print("Random weights:", w.round(3), "| bias:", b)
print("Loss before training:", round(bce_loss(y, y_hat), 4))
print("Accuracy before training:", round(np.mean((y_hat > 0.5) == y), 3))

### 1.3 Training = adjusting weights (gradient descent)
**Learning = adjusting weights through training.** At every step we compute the gradient of the loss and move the weights a small step (the **learning rate**) in the opposite direction:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta\,\frac{\partial L}{\partial \mathbf{w}}, \qquad b \leftarrow b - \eta\,\frac{\partial L}{\partial b}$$

For sigmoid + cross-entropy the gradients are simple: $\frac{\partial L}{\partial \mathbf{w}} = \frac{1}{n}X^\top(\hat{y}-y)$ and $\frac{\partial L}{\partial b} = \frac{1}{n}\sum(\hat{y}-y)$.

In [ ]:
def train_neuron(X, y, lr=1.0, epochs=500, seed=1):
    r = np.random.default_rng(seed)
    w, b = r.normal(0, 1, size=X.shape[1]), 0.0
    history = []
    for epoch in range(epochs):
        y_hat = forward(X, w, b)              # 1) forward pass
        loss = bce_loss(y, y_hat)             # 2) measure error
        dw = X.T @ (y_hat - y) / len(y)       # 3) gradients
        db = np.mean(y_hat - y)
        w -= lr * dw                          # 4) optimizer step
        b -= lr * db
        history.append(loss)
    return w, b, history

w, b, history = train_neuron(X, y, lr=1.0, epochs=500)
acc = np.mean((forward(X, w, b) > 0.5) == y)
print("Learned weights:", w.round(3), "| bias:", round(b, 3))
print(f"Final loss: {history[-1]:.4f} | Training accuracy: {acc:.3f}")

plt.figure(figsize=(5, 3))
plt.plot(history); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("The neuron is learning")
plt.show()

### ✏️ Your turn 1a — Effect of the learning rate
Run the cell below. Which learning rate learns too slowly? Which one is fastest? Then add a much larger value (e.g. `50`) to the list and re-run: does a bigger step always help?

In [ ]:
plt.figure(figsize=(6, 4))
for lr in [0.01, 0.1, 1.0, 5.0]:
    _, _, h = train_neuron(X, y, lr=lr, epochs=500)
    plt.plot(h, label=f"lr = {lr}")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Learning rate comparison")
plt.show()

### ✏️ Your turn 1b — Make a prediction
Use the trained `w` and `b` to predict the pass probability of a student who studies **7 hours** and sleeps **6 hours**. *Hint:* remember to scale the inputs the same way as the training data.

In [ ]:
# TODO: complete this line
student = np.array([7, 6]) / 10.0
prob = None   # replace None with your code

print("Pass probability:", prob)

<details><summary><b>Show solution</b></summary>

```python
student = np.array([7, 6]) / 10.0
prob = forward(student, w, b)
print("Pass probability:", round(float(prob), 3))
```
The model gives a high pass probability because both inputs (especially study hours, which has the larger weight) are high.
</details>

## Part 2 — Why "deep"? (10 min)

A single neuron draws a **straight line** (a linear decision boundary). Many real problems are not linearly separable. Adding **hidden layers** lets the network build more complex representations, one layer on top of the previous one.

We now switch to **Keras**, which computes the gradients for us.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

Xm, ym = make_moons(n_samples=600, noise=0.2, random_state=0)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=0)

plt.figure(figsize=(5, 4))
plt.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap="coolwarm", s=12)
plt.title("Two moons: not separable by a straight line"); plt.show()

In [ ]:
def plot_boundary(model, X, y, ax, title):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict(grid, verbose=0).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=20, cmap="coolwarm", alpha=0.6)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=10, edgecolors="k", linewidths=0.2)
    ax.set_title(title)

def build_model(hidden_layers):
    # hidden_layers: list of neuron counts, e.g. [] (no hidden layer) or [16, 16]
    model = keras.Sequential([keras.Input(shape=(2,))])
    for units in hidden_layers:
        model.add(layers.Dense(units, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
    return model

configs = {"No hidden layer (linear)": [], "Deep: 2 hidden layers [16, 16]": [16, 16]}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, hidden) in zip(axes, configs.items()):
    m = build_model(hidden)
    m.fit(Xm_tr, ym_tr, epochs=150, batch_size=32, verbose=0)
    _, test_acc = m.evaluate(Xm_te, ym_te, verbose=0)
    plot_boundary(m, Xm, ym, ax, f"{name}\ntest accuracy = {test_acc:.2f}")
plt.tight_layout(); plt.show()

### ✏️ Your turn 2 — Explore depth and width
Edit the `configs` dictionary below and re-run. Try for example `[2]`, `[4]`, `[64]`, `[8, 8, 8]`.
- What is the smallest network that still separates the moons well?
- What happens to the boundary with a very large network?

In [ ]:
configs = {"[2]": [2], "[4]": [4], "[64]": [64], "[8, 8, 8]": [8, 8, 8]}   # ← edit me

fig, axes = plt.subplots(1, len(configs), figsize=(4 * len(configs), 3.6))
for ax, (name, hidden) in zip(np.atleast_1d(axes), configs.items()):
    m = build_model(hidden)
    m.fit(Xm_tr, ym_tr, epochs=150, batch_size=32, verbose=0)
    _, test_acc = m.evaluate(Xm_te, ym_te, verbose=0)
    plot_boundary(m, Xm, ym, ax, f"{name}  acc={test_acc:.2f}")
plt.tight_layout(); plt.show()

## Part 3 — The deep learning workflow on MNIST (15 min)

We follow the **basic workflow of DL** from the lecture:

**① Load data → ② Prepare data → ③ Build model → ④ Compile (loss + optimizer) → ⑤ Train → ⑥ Evaluate → ⑦ Predict**

**MNIST** contains 70,000 grayscale images (28×28 pixels) of handwritten digits 0–9.

### ① Load data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print("Training images:", x_train.shape, "| Test images:", x_test.shape)
print("Pixel range:", x_train.min(), "to", x_train.max())

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray"); ax.set_title(y_train[i]); ax.axis("off")
plt.show()

### ② Prepare data
Neural networks train best when inputs are small numbers, so we scale pixels from `0–255` to `0–1`.

In [ ]:
x_train_n = x_train.astype("float32") / 255.0
x_test_n  = x_test.astype("float32") / 255.0

### ③ Build model: a fully-connected (dense) network
`Flatten` turns each 28×28 image into a vector of 784 numbers. Two hidden layers learn features, and the output layer uses **softmax** to give one probability per digit.

In [ ]:
mlp = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),                         # 28x28 -> 784
    layers.Dense(128, activation="relu"),     # hidden layer 1
    layers.Dense(64, activation="relu"),      # hidden layer 2
    layers.Dense(10, activation="softmax"),   # output: 10 class probabilities
], name="MLP")
mlp.summary()

### ✏️ Your turn 3a — Count the parameters by hand
Each `Dense` layer has **(inputs × neurons) weights + (neurons) biases**. Check that your numbers match the `Param #` column above.

In [ ]:
# TODO: fill in the three numbers
layer1 = None   # 784 inputs -> 128 neurons
layer2 = None   # 128 -> 64
layer3 = None   # 64 -> 10
# print("Total:", layer1 + layer2 + layer3)

<details><summary><b>Show solution</b></summary>

```python
layer1 = 784 * 128 + 128   # 100,480
layer2 = 128 * 64 + 64     #   8,256
layer3 = 64 * 10 + 10      #     650
print("Total:", layer1 + layer2 + layer3)   # 109,386
```
Notice that most parameters sit in the first layer, because every pixel connects to every neuron.
</details>

### ④ Compile and ⑤ Train
- **Loss:** `sparse_categorical_crossentropy` (multi-class, integer labels)
- **Optimizer:** `adam` (an improved form of gradient descent)
- **Validation split:** 10% of the training data is held out to monitor generalisation.

In [ ]:
mlp.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_mlp = mlp.fit(x_train_n, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=2)

In [ ]:
def plot_history(h, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, key in zip(axes, ["loss", "accuracy"]):
        ax.plot(h.history[key], "o-", label="train")
        ax.plot(h.history["val_" + key], "o-", label="validation")
        ax.set_xlabel("Epoch"); ax.set_title(f"{title} {key}"); ax.legend()
    plt.tight_layout(); plt.show()

plot_history(hist_mlp, "MLP")

### ⑥ Evaluate on unseen test data

In [ ]:
test_loss, test_acc = mlp.evaluate(x_test_n, y_test, verbose=0)
print(f"MLP test accuracy: {test_acc:.4f}")

### ⑦ Predict, and look at the mistakes

In [ ]:
probs = mlp.predict(x_test_n, verbose=0)
pred = probs.argmax(axis=1)

wrong = np.where(pred != y_test)[0]
print(f"Misclassified: {len(wrong)} of {len(y_test)} test images")

fig, axes = plt.subplots(1, 8, figsize=(12, 1.9))
for ax, idx in zip(axes, wrong[:8]):
    ax.imshow(x_test[idx], cmap="gray"); ax.axis("off")
    ax.set_title(f"true {y_test[idx]}\npred {pred[idx]}", fontsize=9, color="red")
plt.suptitle("Some mistakes: are they hard for humans too?", y=1.08); plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues", colorbar=False)
plt.title("MLP confusion matrix"); plt.show()

### ✏️ Your turn 3b — Draw your own conclusion
Look at the confusion matrix. Which pair of digits does the model confuse most often? Why might that be?

## Part 4 — A small Convolutional Neural Network (10 min)

From the lecture: **CNNs** apply small filters that slide over the image to detect local features (edges, curves). **Pooling** reduces the spatial size so that deeper layers capture higher-level patterns. Unlike `Flatten → Dense`, a CNN keeps the **2-D structure** of the image.

In [ ]:
x_train_c = x_train_n[..., np.newaxis]   # (60000, 28, 28, 1): add a channel dimension
x_test_c  = x_test_n[..., np.newaxis]

cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, kernel_size=3, activation="relu", name="conv1"),   # 16 filters of 3x3
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(32, kernel_size=3, activation="relu", name="conv2"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
], name="CNN")
cnn.summary()

In [ ]:
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_cnn = cnn.fit(x_train_c, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
cnn_acc = cnn.evaluate(x_test_c, y_test, verbose=0)[1]

print(f"\n{'Model':<6} {'Parameters':>12} {'Test accuracy':>15}")
print(f"{'MLP':<6} {mlp.count_params():>12,} {test_acc:>15.4f}   (5 epochs)")
print(f"{'CNN':<6} {cnn.count_params():>12,} {cnn_acc:>15.4f}   (3 epochs)")

### What does the CNN "see"?
Below are the outputs (**feature maps**) of the 16 filters in the first convolutional layer for one test digit. Each filter responds to a different local pattern, such as a stroke direction or an edge.

In [ ]:
feature_extractor = keras.Model(inputs=cnn.inputs, outputs=cnn.get_layer("conv1").output)
fmaps = feature_extractor.predict(x_test_c[:1], verbose=0)[0]     # shape (26, 26, 16)

fig, axes = plt.subplots(2, 9, figsize=(13, 3.2))
axes[0, 0].imshow(x_test[0], cmap="gray"); axes[0, 0].set_title("input")
axes[1, 0].axis("off")
for i in range(16):
    ax = axes[(i // 8), 1 + (i % 8)]
    ax.imshow(fmaps[:, :, i], cmap="viridis"); ax.set_title(f"filter {i}", fontsize=8)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### ✏️ Your turn 4 (bonus) — Shift the digits
The lecture describes CNNs as tolerant to *where* an object appears. Let us test that claim: shift every test image a few pixels to the right and compare how much each model's accuracy drops. Try `shift = 2`, `4` and `6`.

*Discuss:* does the result fully support the claim? (Hint: convolution and pooling give only **partial** shift tolerance; in practice we also train on shifted copies of the images, called **data augmentation**.)

In [ ]:
shift = 2                                                     # ← try 2, 4, 6
x_test_shift = np.roll(x_test_n, shift=shift, axis=2)

mlp_shift = mlp.evaluate(x_test_shift, y_test, verbose=0)[1]
cnn_shift = cnn.evaluate(x_test_shift[..., np.newaxis], y_test, verbose=0)[1]
print(f"MLP: {test_acc:.3f} -> {mlp_shift:.3f} after shifting {shift} px")
print(f"CNN: {cnn_acc:.3f} -> {cnn_shift:.3f} after shifting {shift} px")

## Part 5 — Limitation: small datasets and overfitting (7 min)

From the lecture: deep learning needs **large amounts of data**. With little data the model memorises the training set (including its noise) and **fails to generalise**. This is called **overfitting**.

We train the same MLP architecture on only **300 images** instead of 54,000.

In [ ]:
small_n = 300
x_small, y_small = x_train_n[:small_n], y_train[:small_n]

mlp_small = keras.models.clone_model(mlp)        # same architecture, fresh weights
mlp_small.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist_small = mlp_small.fit(x_small, y_small, epochs=40, batch_size=32,
                           validation_data=(x_test_n[:2000], y_test[:2000]), verbose=0)
plot_history(hist_small, f"MLP on {small_n} images:")
print(f"Final train accuracy:      {hist_small.history['accuracy'][-1]:.3f}")
print(f"Final validation accuracy: {hist_small.history['val_accuracy'][-1]:.3f}")

The **gap** between training and validation accuracy is the signature of overfitting. The slides suggest that traditional ML methods are often a better choice for small datasets. Let us compare with a **Random Forest** trained on the same 300 images.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=0)
rf.fit(x_small.reshape(small_n, -1), y_small)
rf_acc = rf.score(x_test_n[:2000].reshape(2000, -1), y_test[:2000])

print(f"Trained on {small_n} images:")
print(f"  MLP (deep learning) validation accuracy: {hist_small.history['val_accuracy'][-1]:.3f}")
print(f"  Random Forest       validation accuracy: {rf_acc:.3f}")
print(f"Trained on 54,000 images:")
print(f"  MLP (deep learning) test accuracy:       {test_acc:.3f}")

### ✏️ Your turn 5 — How much data is enough?
Change `small_n` in the cell above to `1000`, then `5000`, and re-run both cells. How does the train–validation gap change? At what size does the MLP clearly beat the Random Forest?

## Part 6 — Wrap-up (3 min)

### Quick quiz
1. In Part 1, which line of code is the **optimizer** step?
2. Why could the model with no hidden layer not separate the two moons?
3. Name the seven steps of the DL workflow you followed in Part 3.
4. Why does the CNN usually need fewer parameters than the MLP for similar or better accuracy?
5. Give one situation from the lecture where you would **not** choose deep learning.

<details><summary><b>Show answers</b></summary>

1. `w -= lr * dw` and `b -= lr * db`: the weights move against the gradient, scaled by the learning rate.
2. Without a hidden layer the model can only draw a straight-line boundary, and the moons are not linearly separable. Hidden layers with non-linear activations let the network bend the boundary.
3. Load data → prepare data → build model → compile (loss + optimizer) → train → evaluate → predict.
4. Convolution filters are small (3×3) and **shared** across the whole image, so the same few weights are reused at every position instead of connecting every pixel to every neuron.
5. Any of: real-time decisions on limited hardware; tasks needing explainable reasoning (e.g., legal or medical accountability); ethical or emotional judgement; small or limited datasets; low computational resources.
</details>

### Homework challenge
Replace MNIST with **Fashion-MNIST** (clothing images) by changing one line in Part 3:
```python
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
```
Re-run Parts 3–4. Is the problem harder? Which model wins now? Write 3–4 sentences explaining your findings.

### Programming language note: the same MLP in PyTorch
The concepts are identical across frameworks; only the syntax changes.
```python
import torch.nn as nn
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 10)            # softmax is included in nn.CrossEntropyLoss
)
```